# Evaluate ai.fix_grammar(...) Quality with PySpark

This notebook evaluates grammar correction for minimal edits, meaning preservation, and correctness. The AI transformations and LLM-as-a-Judge evaluation remain in Spark. Only the small, materialized result set is converted to pandas for metrics and charts.

### What You'll Do
1. Correct grammar, spelling, and punctuation in sample text.
2. Score each correction on coherence, consistency, and grammar.
3. Inspect average and per-sample quality.
4. Compare the baseline with a structured custom correction prompt.

### Before You Start
- **Runtime** - This notebook was made for **Fabric 1.3 runtime**.
- **Customize it** - Replace the sample data and adapt the judge criteria to your use case.
- **Keep comparisons fair** - Hold the judge model and prompts fixed while changing one executor setting at a time.
- **Validate important decisions** - LLM judge scores are useful proxies, not a substitute for human-reviewed production samples.

| Metric | Measures |
|--------|----------|
| **Coherence** | The original structure is preserved when possible |
| **Consistency** | Meaning and factual content remain unchanged |
| **Grammar** | Grammar, spelling, and punctuation errors are fixed |

[ai.fix_grammar PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/fix-grammar)


## 1. Setup

Install Pydantic for the structured response schemas used by the judge.
The baseline uses `gpt-5-mini` with low reasoning effort. A fixed
`gpt-5.1` judge scores each quality dimension independently.


In [ ]:
%pip install -q pydantic 2>/dev/null


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from pydantic import BaseModel, Field
from pyspark.sql import functions as F

# Use the smaller model for the function under test and a larger fixed model for judging.
EXECUTOR_OPTIONS = {
    "deploymentName": "gpt-5-mini",  # see https://aka.ms/fabric-ai-models for other models
    "reasoningEffort": "low",
}
JUDGE_OPTIONS = {
    "deploymentName": "gpt-5.1",  # see https://aka.ms/fabric-ai-models for other models
    "reasoningEffort": "medium",
}

class MetricEval(BaseModel):
    reason: str = Field(description="Brief rationale for the score")
    score: int = Field(ge=1, le=5, description="Integer score from 1 to 5")

def materialize(frame):
    cached = frame.cache()
    _ = cached.count()
    error_columns = [name for name in cached.columns if name.endswith("_error")]
    if error_columns:
        has_error = F.lit(False)
        for name in error_columns:
            has_error = has_error | F.coalesce(
                F.length(F.trim(F.col(name).cast("string"))) > 0,
                F.lit(False),
            )
        failed_rows = cached.filter(has_error)
        failure_count = failed_rows.count()
        if failure_count:
            print(f"{failure_count} row(s) contain AI Function errors:")
            id_columns = [
                name for name in ("sample_id", "ticket_id") if name in cached.columns
            ]
            display(failed_rows.select(*id_columns, *error_columns))
    return cached

def fresh_ai_view(frame):
    return frame.select("*")

def add_judge_metric(frame, metric_name, prompt, column_prefix=""):
    score_col = f"{column_prefix}{metric_name}"
    response_col = f"_{score_col}_response"
    raw_score_col = f"_{score_col}_raw_score"
    provider_error_col = f"_{score_col}_error"
    validation_error_col = f"_{score_col}_validation_error"
    judged = fresh_ai_view(frame).ai.generate_response(
        prompt=prompt,
        is_prompt_template=True,
        output_col=response_col,
        error_col=provider_error_col,
        response_format=MetricEval,
        **JUDGE_OPTIONS,
    )
    invalid_score = (
        F.col(raw_score_col).isNull()
        | ~F.col(raw_score_col).between(1, 5)
        | (F.col(raw_score_col) != F.floor(F.col(raw_score_col)))
    )
    validation_message = "Judge score must be an integer from 1 to 5"
    return (
        judged
        .withColumn(
            raw_score_col,
            F.get_json_object(F.col(response_col), "$.score").cast("double"),
        )
        .withColumn(
            validation_error_col,
            F.when(
                invalid_score,
                F.lit(validation_message),
            ).otherwise(F.lit(None).cast("string")),
        )
        .withColumn(
            score_col,
            F.when(
                ~invalid_score,
                F.col(raw_score_col).cast("int"),
            ),
        )
        .withColumn(
            f"{score_col}_reason",
            F.get_json_object(F.col(response_col), "$.reason"),
        )
        .drop(raw_score_col)
    )


## 2. Load Sample Data


In [ ]:
rows = [(1,
  'Dear Support Team, I am writting to inform you that my order #AC-34829 have not arrived yet even though '
  'it was suppose to be delivered last wendsday. I have been waiting for over a week and nobody dont seem to '
  'know where the package is at. Please advise on how to procede with this matter urgently.'),
 (2,
  'Just got the new wireless headphones and honestly there not as good as I was expecting them to be. The '
  'sound quality is ok but the bluetooth keeps disconneting every few minutes which is super anoying. '
  'Definately would not reccommend these to noone who wants reliable audio.'),
 (3,
  "Meeting notes from Tueday's planning session: the team have decided to postpone the product launch untill "
  'Q3 becuase the QA results was not satisfactory and we need to adress several critical bugs before we can '
  'move foreward, also marketing needs more time to finalize there campagin materials and coordinate with '
  'the regional teams.'),
 (4,
  'In my oppinion, the most important factor for economic development in developing countrys are education. '
  'When peoples have access to good schools, they can get better jobs and contributes more to the society. '
  'Goverments should invests more money in education instead of spending it on other less important things.'),
 (5,
  'We are please to present our proposal for the office renovation project; which we believe will '
  'significantly improves employee productivity and moral. The estimated cost of the project are $450000 '
  'over a 18-month timeline, this includes new furniture ergonomic workstations and a redesigned break room '
  'area.')]
df = spark.createDataFrame(rows, ["sample_id", "text"])
display(df)


## 3. Run `ai.fix_grammar`


In [ ]:
corrected_df = materialize(
    df.ai.fix_grammar(
        input_col="text",
        output_col="corrected",
        error_col="executor_error",
        **EXECUTOR_OPTIONS,
    )
)
display(corrected_df.select("text", "corrected"))
display(corrected_df.ai.stats)


## 4. Evaluate with an LLM Judge


In [ ]:
EVAL_METRICS = {
    "coherence": """Score structural preservation from 1 to 5.
A score of 5 means the correction changes only what is necessary and preserves
the original organization and tone.

<original_text>
{text}
</original_text>
<corrected_text>
{corrected}
</corrected_text>""",
    "consistency": """Score meaning preservation from 1 to 5.
A score of 5 means no facts, intent, or content were added, removed, or changed.

<original_text>
{text}
</original_text>
<corrected_text>
{corrected}
</corrected_text>""",
    "grammar": """Score grammar correction quality from 1 to 5.
A score of 5 means grammar, spelling, and punctuation errors are fixed without
introducing new errors.

<original_text>
{text}
</original_text>
<corrected_text>
{corrected}
</corrected_text>""",
}

evaluated_df = fresh_ai_view(corrected_df)
for metric_name, prompt in EVAL_METRICS.items():
    evaluated_df = add_judge_metric(evaluated_df, metric_name, prompt)
evaluated_df = materialize(evaluated_df)
display(evaluated_df.select("text", "corrected", *EVAL_METRICS.keys()))


## 5. Results


In [ ]:
METRICS = ['coherence', 'consistency', 'grammar']
results_pd = evaluated_df.select('sample_id', 'text', 'corrected', 'coherence', 'consistency', 'grammar').toPandas()

score_summary = pd.DataFrame({
    "Metric": ['Coherence', 'Consistency', 'Grammar'],
    "Average score": [results_pd[metric].mean() for metric in METRICS],
    "Scored rows": [results_pd[metric].notna().sum() for metric in METRICS],
})
def quality_status(score, is_complete):
    if not is_complete or pd.isna(score):
        return "INCOMPLETE"
    return "PASS" if score >= 4 else "REVIEW" if score >= 3.5 else "FAIL"

score_summary["Status"] = [
    quality_status(score, scored_rows == len(results_pd))
    for score, scored_rows in zip(
        score_summary["Average score"],
        score_summary["Scored rows"],
    )
]
display(score_summary.round(2))

labels = score_summary["Metric"].tolist()
values = score_summary["Average score"].tolist()
fig = plt.figure(figsize=(13, 4.5))
bar_ax = fig.add_subplot(1, 2, 1)
bars = bar_ax.bar(labels, values, color="#0077aa")
bar_ax.set_ylim(0, 5)
bar_ax.set_ylabel("Score (1-5)")
bar_ax.set_title('Grammar Correction Quality')
bar_ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
bar_ax.tick_params(axis="x", rotation=20)
bar_ax.bar_label(bars, fmt="%.2f", padding=2)

if len(METRICS) >= 3:
    detail_ax = fig.add_subplot(1, 2, 2, polar=True)
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    detail_ax.plot(
        angles + angles[:1],
        values + values[:1],
        "o-",
        linewidth=2,
        color="#9955bb",
    )
    detail_ax.fill(
        angles + angles[:1],
        values + values[:1],
        alpha=0.25,
        color="#9955bb",
    )
    detail_ax.set_xticks(angles)
    detail_ax.set_xticklabels(labels)
    detail_ax.set_ylim(0, 5)
    detail_ax.set_title("Quality Profile", pad=20)
else:
    detail_ax = fig.add_subplot(1, 2, 2)
    detail_ax.hist(
        [results_pd[metric].dropna() for metric in METRICS],
        bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
        label=labels,
        alpha=0.7,
    )
    detail_ax.set_xticks([1, 2, 3, 4, 5])
    detail_ax.set_xlabel("Score")
    detail_ax.set_ylabel("Rows")
    detail_ax.set_title("Score Distribution")
    detail_ax.legend()
plt.tight_layout()
plt.show()

results_pd["scored_metrics"] = results_pd[METRICS].notna().sum(axis=1)
complete_rows = results_pd["scored_metrics"].eq(len(METRICS))
results_pd["average_score"] = (
    results_pd[METRICS].mean(axis=1).where(complete_rows).round(2)
)
results_pd["status"] = [
    quality_status(score, is_complete)
    for score, is_complete in zip(
        results_pd["average_score"],
        complete_rows,
    )
]


In [ ]:
breakdown_pd = results_pd[
    [
        "text",
        "corrected",
        "coherence",
        "consistency",
        "grammar",
        "scored_metrics",
        "average_score",
        "status",
    ]
].copy()
breakdown_pd["text"] = breakdown_pd["text"].str[:100] + "..."
breakdown_pd["corrected"] = breakdown_pd["corrected"].str[:100] + "..."
display(breakdown_pd)


## 6. Optional Refinement: Structured Grammar Correction

Compare `ai.fix_grammar` with a custom response schema that also explains the
most important edits. Both variants use the same judge criteria.


In [ ]:
class GrammarRefinement(BaseModel):
    reason: str = Field(description="Brief explanation of the key corrections")
    corrected_text: str = Field(
        description="Corrected text that preserves meaning and tone"
    )

CUSTOM_PROMPT = """Fix grammar, spelling, and punctuation.
Preserve meaning, tone, proper nouns, and technical terms. Make only necessary edits.

<original_text>
{text}
</original_text>"""

custom_df = fresh_ai_view(corrected_df).ai.generate_response(
    prompt=CUSTOM_PROMPT,
    is_prompt_template=True,
    output_col="_custom_response",
    error_col="_custom_error",
    response_format=GrammarRefinement,
    **EXECUTOR_OPTIONS,
)
custom_df = (
    custom_df
    .withColumn(
        "custom_corrected",
        F.get_json_object(F.col("_custom_response"), "$.corrected_text"),
    )
    .withColumn(
        "custom_reason",
        F.get_json_object(F.col("_custom_response"), "$.reason"),
    )
)
custom_df = materialize(custom_df)

custom_eval_df = fresh_ai_view(custom_df)
for metric_name, prompt in EVAL_METRICS.items():
    custom_eval_df = add_judge_metric(
        custom_eval_df,
        metric_name,
        prompt.replace("{corrected}", "{custom_corrected}"),
        column_prefix="custom_",
    )
custom_eval_df = materialize(custom_eval_df)
display(
    custom_eval_df.select(
        "text", "corrected", "custom_corrected", "custom_reason"
    )
)


In [ ]:
custom_pd = custom_eval_df.select(
    "sample_id", "custom_coherence", "custom_consistency", "custom_grammar"
).toPandas()
comparison_rows_pd = (
    results_pd[["sample_id", "coherence", "consistency", "grammar"]]
    .merge(
        custom_pd,
        on="sample_id",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)
required_columns = [
    "coherence",
    "consistency",
    "grammar",
    "custom_coherence",
    "custom_consistency",
    "custom_grammar",
]
paired_mask = (
    comparison_rows_pd["_merge"].eq("both")
    & comparison_rows_pd[required_columns].notna().all(axis=1)
)
paired_count = int(paired_mask.sum())
excluded_count = int((~paired_mask).sum())
print(f"Paired rows: {paired_count} | Excluded rows: {excluded_count}")
if excluded_count:
    display(
        comparison_rows_pd.loc[
            ~paired_mask,
            ["sample_id", "_merge", *required_columns],
        ]
    )
if not paired_count:
    raise ValueError("No rows have complete baseline and custom grammar scores.")
paired_pd = comparison_rows_pd.loc[paired_mask]

comparison_pd = pd.DataFrame({
    "Metric": ["Coherence", "Consistency", "Grammar", "Overall"],
    "Baseline": [
        paired_pd["coherence"].mean(),
        paired_pd["consistency"].mean(),
        paired_pd["grammar"].mean(),
        paired_pd[["coherence", "consistency", "grammar"]].mean(axis=1).mean(),
    ],
    "Custom": [
        paired_pd["custom_coherence"].mean(),
        paired_pd["custom_consistency"].mean(),
        paired_pd["custom_grammar"].mean(),
        paired_pd[
            ["custom_coherence", "custom_consistency", "custom_grammar"]
        ].mean(axis=1).mean(),
    ],
})
comparison_pd["Delta"] = comparison_pd["Custom"] - comparison_pd["Baseline"]
display(comparison_pd.round(2))

plot_pd = comparison_pd[comparison_pd["Metric"] != "Overall"].set_index("Metric")
ax = plot_pd[["Baseline", "Custom"]].plot.bar(
    figsize=(8, 4),
    color=["#0077aa", "#22cc77"],
    rot=0,
)
ax.set_ylim(0, 5)
ax.set_ylabel("Average score (1-5)")
ax.set_title("Baseline vs Custom Grammar Correction")
ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


## Interpreting Results

| Average score | Suggested action |
|---------------|------------------|
| **4.5-5.0** | Strong candidate for production validation |
| **4.0-4.4** | Good; inspect the lowest-scoring samples |
| **3.5-3.9** | Acceptable for iteration; refine data, prompts, or labels |
| **Below 3.5** | Investigate before broader use |
| **INCOMPLETE** | One or more judge scores are missing; inspect AI Function errors |

| Metric or issue | Likely cause | Next step |
|-----------------|--------------|-----------|
| Coherence | Unnecessary rewriting | Require minimal edits and preserve tone |
| Consistency | Meaning or facts changed | Review the prompt and flagged samples |
| Grammar | Errors remain or new errors appear | Add representative difficult cases |

### Improving Quality

- Treat consistency failures as high priority because the correction changed meaning.
- Use a custom structured prompt when you need an explanation of edits.
- Test a larger executor only after checking whether the prompt is over-correcting.

Keep the judge configuration fixed for comparisons, and confirm release decisions with representative human-reviewed samples.

## Learn More

- [ai.fix_grammar PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/fix-grammar)
- [AI Functions overview](https://learn.microsoft.com/fabric/data-science/ai-functions/overview)
